# Few-shot classifier

Decides whether an Article, or a piece of free text, is about US–Japan fighting in WWII.

1. **Positives**: the hand-picked `config/seed_pages.yaml` Articles plus up to `PER_TAG` in-scope Articles of each
   tag (Ship, Person, Military unit, …), so every kind of Article has examples, not just famous battles.
2. **Negatives**, as many as positives, in three kinds so the model learns more than "war vs. not war":
   - *hard*: left-out Articles closest to the seed pages (airfields, islands, ships, …);
   - *other wars*: battles without the US and Japan, and Second Sino-Japanese War Articles;
   - *far*: left-out Articles not tied to any In-scope Battle.
3. **Text**: the pipeline's cached e5 vectors of `evidence_text` (title + first sentences + every sentence naming a
   theatre or belligerent). For a person or ship the war service is rarely in the first sentences.
4. **Model**: logistic regression on the vectors, related at probability ≥ `THRESHOLD` (0.70, set for precision;
   see section 3). Free text has its own `TEXT_THRESHOLD` and must also be at least `FLOOR`-similar to some positive
   (section 4).

Measured on the human labels (`gray_labels.yaml`, `eval_set.yaml`), none of which is ever an example or used for
tuning. Runs **offline** from `Data/interim/` written by `crawl_corpus.ipynb`.

**Caveat:** the extra positives and negatives are the pipeline's own decisions, so the model learns to *imitate the
pipeline*. That carries the pipeline's judgement over to free text (where rules have no links or Wikidata to read),
but it cannot beat the pipeline on Articles, and its disagreements with it are as likely its mistakes as the pipeline's.

In [1]:
import collections
import csv
import json
import random
import re
import sqlite3
import sys
from pathlib import Path

import mwparserfromhell
import numpy as np
import torch
import yaml
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

ROOT = Path.cwd().parent if Path.cwd().name == "Processing" else Path.cwd()
CONFIG_DIR = ROOT / "Processing" / "config"
INTERIM = ROOT / "Data" / "interim"
PREDICTIONS_PATH = INTERIM / "few_shot_predictions.csv"
sys.path.insert(0, str(ROOT / "Processing"))
from evidence import REF_RE

E5_MODEL = "intfloat/e5-base-v2"
E5_REVISION = "f52bf8ec8c7124536f0efb74aca902b2995e5bcd"  # must match crawl_corpus.ipynb
E5_VARIANT = "intfloat/e5-base-v2|war-sentences-v1"  # cached evidence_text vectors written by crawl_corpus.ipynb
PER_TAG = 100  # in-scope Articles sampled per tag as extra positives
C_GRID = [0.1, 0.3, 1, 3, 10]
SEED = 0
THRESHOLD = 0.70  # set by hand for precision; see section 3

db = sqlite3.connect(f"file:{INTERIM / 'cache.sqlite'}?mode=ro", uri=True)
vectors = {t: np.frombuffer(v, dtype=np.float32)
           for t, v in db.execute("SELECT title, vec FROM vectors WHERE variant=?", (E5_VARIANT,))}
rows = {}
for name in ("tagged.jsonl", "excluded.jsonl", "untagged.jsonl"):
    with (INTERIM / name).open() as f:
        for line in f:
            r = json.loads(line)
            rows[r["title"]] = r

_model = None


def embed(texts):
    """Free text, embedded like the cached Article vectors (e5 passage marker)."""
    global _model
    if _model is None:
        device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
        _model = SentenceTransformer(E5_MODEL, revision=E5_REVISION, device=device)
    return _model.encode([f"passage: {t}" for t in texts], normalize_embeddings=True, show_progress_bar=False)


print(f"{len(rows):,} classified Articles · {len(vectors):,} cached vectors")

/Users/nivgreenstein/Desktop/projects/geoai/Corpus2GeoRAG_Pacific/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


14,665 classified Articles · 14,672 cached vectors


## 1. Labels kept apart from the examples

In [2]:
gray = yaml.safe_load((CONFIG_DIR / "gray_labels.yaml").read_text())
eval_cfg = yaml.safe_load((CONFIG_DIR / "eval_set.yaml").read_text())
audit_cfg = yaml.safe_load((CONFIG_DIR / "corpus_audit.yaml").read_text()) or {}

calibration = {t: bool(y) for t, y in gray["calibration"].items()}
holdout = {t: bool(y) for t, y in gray["holdout"].items()}
evaluation = {t: y in ("keep", "marginal")  # marginal counts as correct (eval_set.yaml, 2026-10-17)
              for section in eval_cfg.values() if isinstance(section, dict) for t, y in section.items() if y}
audit = {t for section in audit_cfg.values() if isinstance(section, dict) for t in section}
never = set(calibration) | set(holdout) | set(evaluation) | audit
print(f"calibration {len(calibration)} · holdout {len(holdout)} · eval_set {len(evaluation)} · "
      f"{len(never):,} titles never used as examples")

calibration 76 · holdout 68 · eval_set 365 · 658 titles never used as examples


## 2. Examples

In [3]:
seed_cfg = yaml.safe_load((CONFIG_DIR / "seed_pages.yaml").read_text())
seeds = [t for titles in seed_cfg.values() for t in titles]
assert not set(seeds) & never, f"seed pages that are also labels: {set(seeds) & never}"
missing = [t for t in seeds if t not in vectors]
if missing:
    print(f"seed pages without a cached vector, skipped: {missing}")
seeds = [t for t in seeds if t in vectors]

usable = [t for t in rows if t in vectors and t not in never]
by_tag = collections.defaultdict(list)
for t in usable:
    if rows[t]["in_scope"] and t not in seeds:
        by_tag[rows[t]["primary_tag"] or "—"].append(t)
rng = random.Random(SEED)
extra = [t for tag in sorted(by_tag) for t in rng.sample(by_tag[tag], min(PER_TAG, len(by_tag[tag])))]
positives = seeds + extra

# hard: the 10% of left-out Articles closest to a seed page, sampled (the very closest are the likeliest to be
# Articles the pipeline wrongly left out)
out_pool = sorted(t for t in usable if not rows[t]["in_scope"])
S = np.stack([vectors[t] for t in seeds])
best = dict(zip(out_pool, (np.stack([vectors[t] for t in out_pool]) @ S.T).max(1)))
cut = np.quantile(list(best.values()), 0.9)
OTHER_WAR_REASONS = ("battle without the US and Japan", "umbrella only: Second Sino-Japanese War")
n = len(positives) // 3
hard = rng.sample([t for t in out_pool if best[t] >= cut], n)
other_wars = rng.sample([t for t in out_pool if rows[t]["reason"].startswith(OTHER_WAR_REASONS) and t not in hard], n)
far = rng.sample([t for t in out_pool if rows[t]["reason"] == "not tied to any in-scope battle"
                  and t not in hard + other_wars], n)

examples = {"seed": seeds, "in-scope sample": extra, "hard": hard, "other wars": other_wars, "far": far}
ex_titles = [t for titles in examples.values() for t in titles]
ex_kinds = np.array([kind for kind, titles in examples.items() for _ in titles])
X = np.stack([vectors[t] for t in ex_titles])
y = np.isin(ex_kinds, ["seed", "in-scope sample"])
for kind, titles in examples.items():
    print(f"{kind:16} {len(titles):5}  e.g. {', '.join(titles[:3])}")
print(f"\npositives by tag: {dict(collections.Counter(rows[t]['primary_tag'] or '—' for t in extra))}")

seed                55  e.g. Attack on Pearl Harbor, Battle of Wake Island, Battle of Bataan
in-scope sample   1020  e.g. Nakajima Ki-115 Tsurugi, North American P-51 Mustang, Nakajima C6N Saiun
hard               358  e.g. USS Sproston (DD-577), Naval Air Facility Atsugi, Sheng Shicai
other wars         358  e.g. Type 96 light machine gun, Battle of Shumshu, Du Zhongyuan
far                358  e.g. USS Vulcan (AR-5), Roekiah, Jingei-class submarine tender

positives by tag: {'Aircraft': 55, 'Battle': 100, 'Campaign': 27, 'Diplomacy/Document': 60, 'Event': 43, 'Intelligence': 19, 'Location': 100, 'Military unit': 100, 'Operation': 77, 'Organization': 21, 'Person': 100, 'Ship': 100, 'Ship class': 100, 'Weapon/Equipment': 89, '—': 29}


## 3. Choose C and the threshold by cross-validation on the examples
Out-of-fold probabilities (5 folds) for each regularisation strength `C`; the F1-best threshold is printed for
reference. The threshold actually used is `THRESHOLD`, set by hand for precision (2026-10-19): at 0.7 the human labels
gave 95% precision and 44% recall, against 89% / 81% at the cross-validated 0.40. Because it was picked by looking at
the human labels, section 5's scores at this threshold are slightly optimistic.

In [4]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)


def f1(pred, labels):
    tp, fp, fn = (pred & labels).sum(), (pred & ~labels).sum(), (~pred & labels).sum()
    return 2 * tp / max(2 * tp + fp + fn, 1)


cv = {}
for c in C_GRID:
    lr = LogisticRegression(C=c, class_weight="balanced", max_iter=5000)
    p = cross_val_predict(lr, X, y, cv=folds, method="predict_proba")[:, 1]
    cut_f1 = max((f1(p >= th, y), th) for th in np.arange(0.2, 0.81, 0.02))
    cv[c] = (*cut_f1, p)
    print(f"C={c:<4} out-of-fold F1 {cut_f1[0]:.3f} at threshold {cut_f1[1]:.2f}")
C_BEST = max(cv, key=lambda c: cv[c][0])
_, cv_threshold, oof = cv[C_BEST]
print(f"\nchosen C = {C_BEST} · F1-best threshold {cv_threshold:.2f} · used THRESHOLD {THRESHOLD:.2f}")
for kind in examples:
    m = ex_kinds == kind
    print(f"  out-of-fold on {kind:16} {((oof[m] >= THRESHOLD) == y[m]).mean():.0%} correct")

clf = LogisticRegression(C=C_BEST, class_weight="balanced", max_iter=5000).fit(X, y)

C=0.1  out-of-fold F1 0.792 at threshold 0.48
C=0.3  out-of-fold F1 0.805 at threshold 0.46
C=1    out-of-fold F1 0.819 at threshold 0.42


C=3    out-of-fold F1 0.822 at threshold 0.40


C=10   out-of-fold F1 0.826 at threshold 0.40

chosen C = 10 · F1-best threshold 0.40 · used THRESHOLD 0.70
  out-of-fold on seed             87% correct
  out-of-fold on in-scope sample  63% correct
  out-of-fold on hard             82% correct
  out-of-fold on other wars       94% correct
  out-of-fold on far              95% correct


## 4. Free text: its own threshold and floor
Free text is a sentence or a paragraph, not the title + war sentences the model was trained on, so its probabilities
run lower and the Article `THRESHOLD` would reject most of it. To set its cut-offs without any human label, every
example Article is also embedded as free text (the first three sentences of its lead) and scored by a model trained on
the *other* folds:
- `TEXT_THRESHOLD`: the F1-best threshold on those leads;
- `FLOOR`: the similarity to the nearest *other* positive that 99% of positive leads reach. Free text below it is unlike
  every example and is not related, whatever the probability.

In [5]:
def lead_text(title):
    """First three sentences of the Article's lead: a stand-in for free text about it."""
    row = db.execute("SELECT wikitext FROM wikitext WHERE title=?", (title,)).fetchone()
    if not row:
        return None
    lead = mwparserfromhell.parse(row[0].split("\n==", 1)[0]).strip_code()
    text = re.sub(r"\s+", " ", REF_RE.sub("", lead)).strip()
    return ".".join(text.split(".")[:3]).strip() or None


ex_leads = [lead_text(t) for t in ex_titles]
has_lead = np.array([bool(t) for t in ex_leads])
L = np.zeros_like(X)
L[has_lead] = embed([t for t in ex_leads if t])

lead_prob = np.full(len(ex_titles), np.nan)
for train, test in folds.split(X, y):
    test = test[has_lead[test]]
    fold_clf = LogisticRegression(C=C_BEST, class_weight="balanced", max_iter=5000).fit(X[train], y[train])
    lead_prob[test] = fold_clf.predict_proba(L[test])[:, 1]

ly, lp = y[has_lead], lead_prob[has_lead]
print(" thr  precision recall  F1   (example leads, out-of-fold)")
best_f1 = []
for th in np.round(np.arange(0.10, 0.81, 0.05), 2):
    pred = lp >= th
    tp, fp, fn = (pred & ly).sum(), (pred & ~ly).sum(), (~pred & ly).sum()
    best_f1.append((f1(pred, ly), th))
    print(f" {th:.2f}  {tp / max(tp + fp, 1):6.0%}  {tp / max(tp + fn, 1):6.0%}  {f1(pred, ly):.2f}")
TEXT_THRESHOLD = float(max(best_f1)[1])

pos = np.where(y & has_lead)[0]
sims = L[pos] @ X[y].T
sims[np.arange(len(pos)), np.searchsorted(np.where(y)[0], pos)] = -np.inf  # not its own Article
FLOOR = float(np.quantile(sims.max(1), 0.01))
print(f"\nTEXT_THRESHOLD = {TEXT_THRESHOLD:.2f} (Articles: THRESHOLD = {THRESHOLD:.2f}) · FLOOR = {FLOOR:.3f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16178.84it/s]

 thr  precision recall  F1   (example leads, out-of-fold)
 0.10     62%     94%  0.74
 0.15     65%     90%  0.76
 0.20     69%     85%  0.76
 0.25     71%     81%  0.76
 0.30     75%     77%  0.76
 0.35     78%     74%  0.76
 0.40     80%     69%  0.74
 0.45     80%     66%  0.73
 0.50     83%     63%  0.71
 0.55     85%     57%  0.68
 0.60     87%     51%  0.65
 0.65     89%     47%  0.61
 0.70     91%     42%  0.57
 0.75     91%     35%  0.51
 0.80     94%     28%  0.43

TEXT_THRESHOLD = 0.30 (Articles: THRESHOLD = 0.70) · FLOOR = 0.781


## 5. Measure on the human labels
None of these Articles is an example and none was used for tuning. The pipeline's own decision on the same Articles is
the reference to beat.

In [6]:
def report(name, pred, labels):
    labels = np.asarray(labels, dtype=bool)
    tp, fp = int((pred & labels).sum()), int((pred & ~labels).sum())
    fn, tn = int((~pred & labels).sum()), int((~pred & ~labels).sum())
    print(f"{name:34} n={len(labels):4}  accuracy {(tp + tn) / len(labels):.0%} · precision {tp / max(tp + fp, 1):.0%}"
          f" · recall {tp / max(tp + fn, 1):.0%} · F1 {f1(pred, labels):.2f}")


def predict_articles(titles):
    return clf.predict_proba(np.stack([vectors[t] for t in titles]))[:, 1] >= THRESHOLD


for name, labels in (("calibration", calibration), ("holdout", holdout), ("eval_set (keep+marginal)", evaluation),
                     ("all", {**calibration, **holdout, **evaluation})):
    titles = [t for t in labels if t in vectors]
    truth = [labels[t] for t in titles]
    report(name, predict_articles(titles), truth)
    report("  pipeline on the same Articles", np.array([bool(rows[t]["in_scope"]) for t in titles]), truth)

calibration                        n=  76  accuracy 70% · precision 91% · recall 31% · F1 0.47
  pipeline on the same Articles    n=  76  accuracy 92% · precision 93% · recall 88% · F1 0.90
holdout                            n=  68  accuracy 72% · precision 92% · recall 38% · F1 0.54
  pipeline on the same Articles    n=  68  accuracy 90% · precision 100% · recall 76% · F1 0.86
eval_set (keep+marginal)           n= 305  accuracy 59% · precision 95% · recall 47% · F1 0.63
  pipeline on the same Articles    n= 305  accuracy 85% · precision 91% · recall 88% · F1 0.90
all                                n= 449  accuracy 63% · precision 95% · recall 44% · F1 0.60
  pipeline on the same Articles    n= 449  accuracy 87% · precision 92% · recall 87% · F1 0.90


## 6. Classify free text
Free text is embedded on the fly and judged with `TEXT_THRESHOLD` and `FLOOR` (section 4). First a check on the human
labels in free-text form: each labelled Article's first three lead sentences, classified like any free text.

In [7]:
def text_scores(V):
    return clf.predict_proba(V)[:, 1], (V @ X[y].T).max(1)


labels = {**calibration, **holdout, **evaluation}
lab_titles = [t for t in labels if lead_text(t)]
lab_prob, lab_closest = text_scores(embed([lead_text(t) for t in lab_titles]))
lab_truth = np.array([labels[t] for t in lab_titles])
report("labelled leads as free text", (lab_prob >= TEXT_THRESHOLD) & (lab_closest >= FLOOR), lab_truth)
report("  same, with the Article THRESHOLD", (lab_prob >= THRESHOLD) & (lab_closest >= FLOOR), lab_truth)

labelled leads as free text        n= 509  accuracy 63% · precision 79% · recall 51% · F1 0.62
  same, with the Article THRESHOLD n= 509  accuracy 49% · precision 86% · recall 16% · F1 0.27


In [8]:
def classify_text(text, verbose=True):
    """True when `text` is about US–Japan fighting in WWII."""
    v = embed([text])[0]
    (prob,), (closest,) = text_scores(v[None])
    if verbose:
        nearest = np.argsort(-(X @ v))[:5]
        print(f"probability {prob:.2f} (threshold {TEXT_THRESHOLD:.2f}) · closest positive {closest:.3f} "
              f"(floor {FLOOR:.3f}) · nearest: " + "; ".join(f"{ex_titles[i]} ({ex_kinds[i]})" for i in nearest))
    return bool(prob >= TEXT_THRESHOLD and closest >= FLOOR)


tests = {
    "Midway": "The Battle of Midway was a pivotal naval battle in the Pacific Theater of World War II, fought between "
              "the United States and Japan in June 1942. The U.S. Navy, with the help of codebreakers, was able to "
              "anticipate and counter the Japanese attack, leading to a decisive victory.",
    "Israel–Palestine": "The conflict between Israel and Palestine has been ongoing for decades, with both sides "
                        "claiming historical and religious rights to the land. The situation is complex, involving "
                        "political, social, and humanitarian issues that have led to numerous wars and peace negotiations.",
    "Stalingrad": "The Battle of Stalingrad was a major battle on the Eastern Front of World War II, in which Germany "
                  "and its allies fought the Soviet Union for control of the city of Stalingrad.",
    "Eiffel Tower": "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It was "
                    "designed by Gustave Eiffel and completed in 1889.",
    "Destroyer": "USS Johnston was a Fletcher-class destroyer of the United States Navy. She was sunk by Japanese "
                 "battleships and cruisers in the Battle off Samar in October 1944.",
}
for name, text in tests.items():
    print(f"\n{name}:")
    print("  →", classify_text(text))


Midway:
probability 0.62 (threshold 0.30) · closest positive 0.899 (floor 0.781) · nearest: Battle of Midway (seed); Battle of Midway order of battle (in-scope sample); Midway (1964 game) (in-scope sample); Battle of the Philippine Sea (seed); Battle of the Coral Sea (seed)
  → True

Israel–Palestine:
probability 0.01 (threshold 0.30) · closest positive 0.724 (floor 0.781) · nearest: Battle of Kissoué (other wars); Battle of Jezzine (1941) (other wars); Battle of the Litani River (other wars); Battle of Merdjayoun (other wars); Battle of Damour (other wars)
  → False

Stalingrad:
probability 0.17 (threshold 0.30) · closest positive 0.797 (floor 0.781) · nearest: Battle of Kissoué (other wars); Battle of the Litani River (other wars); Vasily Chuikov (hard); Battle of Damour (other wars); Battle of Manila (1945) (seed)
  → False

Eiffel Tower:
probability 0.00 (threshold 0.30) · closest positive 0.742 (floor 0.781) · nearest: Japanese battleship Mikasa (hard); Chinese ironclad Zhenyuan 

## 7. Classify every scraped Article
Every crawled Article with a cached vector, except the examples (the model has seen their labels). Written to
`Data/interim/few_shot_predictions.csv`, with the human label where there is one.

In [9]:
used = set(ex_titles)
all_titles = [t for t in rows if t in vectors and t not in used]
prob_all = clf.predict_proba(np.stack([vectors[t] for t in all_titles]))[:, 1]

preds = []
for t, p in zip(all_titles, prob_all):
    r = rows[t]
    preds.append({"title": t, "primary_tag": r["primary_tag"], "pipeline": bool(r["in_scope"]),
                  "decided_by": r["decided_by"], "few_shot": bool(p >= THRESHOLD), "probability": round(float(p), 3),
                  "label": evaluation.get(t, holdout.get(t, calibration.get(t)))})
with PREDICTIONS_PATH.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(preds[0]))
    w.writeheader()
    w.writerows(preds)
print(f"classified {len(preds):,} Articles ({sum(t not in vectors for t in rows):,} without a vector skipped) "
      f"→ {sum(p['few_shot'] for p in preds):,} related · wrote {PREDICTIONS_PATH.name}")

classified 12,515 Articles (1 without a vector skipped) → 2,447 related · wrote few_shot_predictions.csv


## 8. Agreement with the pipeline
The pipeline's decision is a **reference, not ground truth**. Where the two disagree, the human labels (section 5)
say who tends to be right; this section shows *where* they disagree.

In [10]:
def agreement(group):
    pipe_y = np.array([p["pipeline"] for p in group]); few_y = np.array([p["few_shot"] for p in group])
    both, only_pipe, only_few = int((pipe_y & few_y).sum()), int((pipe_y & ~few_y).sum()), int((~pipe_y & few_y).sum())
    return {"n": len(group), "agree": (pipe_y == few_y).mean(), "both in": both, "pipeline only": only_pipe,
            "few-shot only": only_few, "recall vs pipeline": both / max(both + only_pipe, 1),
            "precision vs pipeline": both / max(both + only_few, 1)}


def table(key, min_n=50):
    groups = collections.defaultdict(list)
    for p in preds:
        groups[p[key] or "—"].append(p)
    print(f"{key:20} {'n':>6} {'agree':>6} {'both in':>8} {'pipe only':>10} {'few only':>9} {'recall':>7} {'prec':>6}")
    for name, group in sorted(groups.items(), key=lambda kv: -len(kv[1])):
        if len(group) >= min_n:
            a = agreement(group)
            print(f"{str(name)[:20]:20} {a['n']:6} {a['agree']:6.0%} {a['both in']:8} {a['pipeline only']:10} "
                  f"{a['few-shot only']:9} {a['recall vs pipeline']:7.0%} {a['precision vs pipeline']:6.0%}")


a = agreement(preds)
print(f"overall: {a['agree']:.0%} agree · few-shot finds {a['recall vs pipeline']:.0%} of the pipeline's corpus · "
      f"{a['precision vs pipeline']:.0%} of its picks are in the corpus\n")
table("primary_tag")
print()
table("decided_by", min_n=1)

overall: 76% agree · few-shot finds 44% of the pipeline's corpus · 83% of its picks are in the corpus

primary_tag               n  agree  both in  pipe only  few only  recall   prec
Ship                   5136    66%      993       1623        99     38%    91%
Person                 3746    79%      553        771        26     42%    96%
Military unit           957    85%      169         89        54     66%    76%
Location                835    83%      158         68        70     70%    69%
Aircraft                297    98%        4          1         5     80%    44%
Battle                  272    86%      101         25        12     80%    89%
Media                   266    86%        0          0        36      0%     0%
Ship class              235    88%       28         14        15     67%    65%
Weapon/Equipment        174    75%        2          1        43     67%     4%
—                       135    83%        3          0        23    100%    12%
Event            

In [11]:
def show(title, items, n=15):
    print(title)
    for p in items[:n]:
        print(f"  probability {p['probability']:.2f} · {p['primary_tag'] or '—':16} {p['title']}")


show("Few-shot says related, pipeline left out (most confident first):",
     sorted((p for p in preds if p["few_shot"] and not p["pipeline"]), key=lambda p: -p["probability"]))
print()
show("Pipeline kept, few-shot says unrelated (most confident first):",
     sorted((p for p in preds if p["pipeline"] and not p["few_shot"]), key=lambda p: p["probability"]))

Few-shot says related, pipeline left out (most confident first):
  probability 0.98 · Diplomacy/Document List of weapons of World War II Japanese aircraft
  probability 0.98 · Ship class       Armament of the Iowa-class battleship
  probability 0.97 · —                Japanese invasion money
  probability 0.97 · Media            Attack on Pearl Harbor in popular culture
  probability 0.97 · Diplomacy/Document List of Japanese World War II army bombs
  probability 0.97 · —                Attacks on Australia during World War II
  probability 0.96 · Diplomacy/Document List of World War II weapons of the United States
  probability 0.96 · —                Evacuations of civilians in Japan during World War II
  probability 0.96 · Location         Imperial Japanese Navy bases and facilities
  probability 0.95 · Media            The Fighting Seabees
  probability 0.95 · Event            Japanese government–issued Oceanian pound
  probability 0.95 · Media            Guadalcanal Diary (film)
 